In [2]:
import pandas as pd
import numpy as np

tx_cols = ["TransactionID", "isFraud", "TransactionDT", "TransactionAmt",
           "ProductCD", "card1", "card2", "card3", "card4", "card5", "card6",
           "addr1", "dist1", "P_emaildomain", "R_emaildomain"]
id_cols = ["TransactionID", "DeviceType", "DeviceInfo"]

tx = pd.read_csv("data/train_transaction.csv", usecols=tx_cols)
idt = pd.read_csv("data/train_identity.csv", usecols=id_cols)
df = tx.merge(idt, on="TransactionID", how="left")

# sort by time so "before" really means before
df = df.sort_values("TransactionDT").reset_index(drop=True)
print(df.shape)

(590540, 17)


In [3]:
df["hour"] = (df["TransactionDT"] // 3600) % 24
df["day_of_week"] = (df["TransactionDT"] // 86400) % 7

df["card_key"] = (df["card1"].astype(str) + "_" + df["card2"].astype(str) + "_" +
                  df["card3"].astype(str) + "_" + df["card5"].astype(str))

print(df["card_key"].nunique(), "unique cards")

13656 unique cards


In [4]:
g = df.groupby("card_key")

# velocity
df["tx_count_before"] = g.cumcount()
df["secs_since_last"] = g["TransactionDT"].diff()

# amount compared with this card's usual amount (past only)
past_sum = g["TransactionAmt"].cumsum() - df["TransactionAmt"]
df["avg_amt_before"] = past_sum / df["tx_count_before"].replace(0, np.nan)
df["amt_vs_avg"] = df["TransactionAmt"] / df["avg_amt_before"]

# did something change since the last transaction of this card?
for col, new in [("DeviceInfo", "device_changed"),
                 ("addr1", "addr_changed"),
                 ("P_emaildomain", "email_changed")]:
    prev = g[col].shift(1)
    df[new] = ((df[col] != prev) & df[col].notna() & prev.notna()).astype(int)

print(df[["tx_count_before", "secs_since_last", "amt_vs_avg",
          "device_changed", "addr_changed", "email_changed"]].describe())

       tx_count_before  secs_since_last     amt_vs_avg  device_changed  \
count    579507.000000     5.658510e+05  565851.000000   590540.000000   
mean       1154.012952     1.704539e+05       1.087552        0.063186   
std        1992.573749     7.081915e+05       1.730848        0.243298   
min           0.000000     0.000000e+00       0.002841        0.000000   
25%          41.000000     9.800000e+02       0.391895        0.000000   
50%         292.000000     5.920000e+03       0.659334        0.000000   
75%        1318.000000     5.826900e+04       1.158687        0.000000   
max       14111.000000     1.559967e+07     145.944802        1.000000   

        addr_changed  email_changed  
count  590540.000000  590540.000000  
mean        0.448185       0.401238  
std         0.497308       0.490149  
min         0.000000       0.000000  
25%         0.000000       0.000000  
50%         0.000000       0.000000  
75%         1.000000       1.000000  
max         1.000000       1.

In [5]:
for c in ["device_changed", "addr_changed", "email_changed"]:
    print(df.groupby(c)["isFraud"].agg(["mean", "count"]))
    print()

df["fast"] = (df["secs_since_last"] < 60).astype(int)
print(df.groupby("fast")["isFraud"].agg(["mean", "count"]))

                    mean   count
device_changed                  
0               0.032997  553226
1               0.064533   37314

                  mean   count
addr_changed                  
0             0.044816  325869
1             0.022893  264671

                   mean   count
email_changed                  
0              0.037786  353593
1              0.030817  236947

          mean   count
fast                  
0     0.034048  571510
1     0.063269   19030


In [7]:
df["event_timestamp"] = pd.Timestamp("2017-12-01") + pd.to_timedelta(df["TransactionDT"], unit="s")

feature_cols = ["TransactionID", "card_key", "event_timestamp", "hour", "day_of_week",
                "tx_count_before", "secs_since_last", "amt_vs_avg",
                "device_changed", "addr_changed", "email_changed"]

df[feature_cols + ["isFraud"]].to_parquet("data/features.parquet", index=False)
print("saved")

saved
